# Unsloth Deep Dive #01 — Internals

How does Unsloth deliver 2x speed + 70% less VRAM? This notebook **pops the hood and looks inside**.

## What This Notebook Gives You

1. **A map of the architecture** — how the package is organized
2. **The patching mechanism** — why `import unsloth` has to come first
3. **Triton kernels** — which ones, what they do, why they're fast
4. **Manual backprop** — why Unsloth bypasses `autograd`
5. **Model registry** — how each model is detected
6. **Memory savings** — the math of fused operations

This knowledge is a lifesaver when debugging. Reading an error message and being able to say "ah, this is happening during patching" is worth a lot.

## 1. Package Organization — High-Level Map

Unsloth consists of two packages:

```
unsloth/                          ← high-level API + model wrappers
  ├── __init__.py                 ← patching trigger
  ├── models/                     ← model-specific code
  │   ├── loader.py               ← FastModel, FastLanguageModel, FastVisionModel
  │   ├── llama.py                ← FastLlamaModel (legacy)
  │   ├── qwen3.py, qwen3_moe.py  ← model-specific patches
  │   ├── mistral.py, gemma.py
  │   ├── vision.py               ← VLM model handling
  │   └── _utils.py
  ├── kernels/                    ← Triton kernels (where the real work lives)
  │   ├── cross_entropy_loss.py   ← 463 lines
  │   ├── fast_lora.py            ← 730 lines (LoRA forward+backward)
  │   ├── rope_embedding.py       ← 465 lines
  │   ├── rms_layernorm.py        ← 339 lines
  │   ├── swiglu.py               ← 143 lines
  │   ├── geglu.py                ← 290 lines
  │   ├── layernorm.py            ← 227 lines
  │   ├── flex_attention.py       ← 187 lines
  │   ├── fp8.py                  ← 624 lines (FP8 quantization)
  │   └── moe/                    ← MoE kernels (new in 2026)
  ├── trainer.py                  ← UnslothTrainer (for CPT)
  ├── save.py                     ← GGUF/merged save
  └── tokenizer_utils.py

unsloth_zoo/                      ← shared utility functions
  ├── compiler.py                 ← model compile/patch logic
  ├── patch_torch_functions.py    ← PyTorch function overrides
  ├── patching_utils.py           ← general patch helpers
  ├── gradient_checkpointing.py   ← Unsloth-specific checkpointing
  ├── fused_losses/               ← fused loss kernels
  ├── flex_attention/             ← flex attention impl
  ├── peft_utils.py               ← PEFT (LoRA) integration
  ├── temporary_patches/          ← temporary fixes
  ├── rl_replacements.py          ← RL trainer patches
  └── vllm_*.py                   ← vLLM integration
```

**Total Triton kernel code:** ~4600 lines. This isn't "just a library" — it's a deeply optimized full GPU program.

In [ ]:
# Inspect the package layout
import unsloth, os

unsloth_dir = os.path.dirname(unsloth.__file__)
print(f'Unsloth dir: {unsloth_dir}\n')

# List the kernel files
kernel_dir = os.path.join(unsloth_dir, 'kernels')
if os.path.exists(kernel_dir):
    print('=== Triton Kernels ===')
    for f in sorted(os.listdir(kernel_dir)):
        if f.endswith('.py'):
            path = os.path.join(kernel_dir, f)
            with open(path) as fh:
                lines = sum(1 for _ in fh)
            print(f'  {f:30s} {lines:5d} lines')

## 2. The Patching Mechanism — Why `import unsloth` Goes First

### The Problem

Unsloth swaps out **functions and classes from the `transformers` library at runtime**. For example:

- `LlamaRotaryEmbedding.forward` → its own Triton-based version
- `LlamaMLP.forward` → fused SwiGLU kernel
- `LlamaRMSNorm.forward` → fused RMSNorm kernel
- Cross-entropy loss → fused CE kernel

This is called **monkey patching**. The catch: monkey patching can't change references that were captured **before** the patch was applied.

```python
# WRONG order
from transformers import LlamaForCausalLM   # 1. transformers loaded
from transformers.models.llama.modeling_llama import LlamaRMSNorm
import unsloth                                # 2. patches now applied
                                               # → LlamaRMSNorm got patched, BUT
                                               # → we already captured the old reference above!
model = LlamaForCausalLM.from_pretrained(...)  # 3. uses the un-patched version
```

### Correct Order

```python
import unsloth                  # 1. FIRST — register the patches
from transformers import ...     # 2. when transformers is imported, the patches kick in
model = ...                      # 3. uses the patched version
```

Unsloth also checks at import time whether `transformers`, `peft`, or `bitsandbytes` are already loaded. If they are, it **warns**:

```
Unsloth: Should be imported before trl, transformers, peft.
Your code may run slower or encounter memory issues.
```

### Patch Types

Unsloth patches at two levels:

**1. Import-time patches** (`unsloth_zoo.patching_utils`)
- Override `transformers.modeling_utils.PreTrainedModel.from_pretrained`
- Patch `peft.tuners.lora.LoraLayer`
- Modify low-level CUDA calls in `bitsandbytes`

**2. Load-time patches** (`unsloth/models/*.py`)
- After the model is loaded, apply class-specific patches
- Example: when `LlamaForCausalLM` is loaded, `models/llama.py` becomes active
- `LlamaDecoderLayer.forward` gets swapped for a Triton version

In [ ]:
# Have the patched functions' source files actually changed?
import unsloth
from transformers.models.llama.modeling_llama import LlamaRMSNorm
from transformers.models.qwen3.modeling_qwen3 import Qwen3RMSNorm

# Patched forward methods now live in unsloth/kernels/*
import inspect
for cls in [LlamaRMSNorm, Qwen3RMSNorm]:
    fwd = cls.forward
    src_file = inspect.getsourcefile(fwd) or '?'
    is_patched = 'unsloth' in src_file.lower()
    print(f'{cls.__name__:20s} forward → {src_file}  (patched: {is_patched})')

## 3. Triton Kernels — The Real Work

Triton is OpenAI's **GPU programming language**. Instead of CUDA C, you write Python-style code and the compiler emits optimized CUDA. Unsloth's performance secret: rewrite the critical operations in Triton and bypass PyTorch's eager mode.

### The Most Important Kernels

**`fast_lora.py` (730 lines) — LoRA forward + backward**

In standard PEFT, LoRA looks like:
```
y = x @ W_base + (x @ A @ B) * scale
```
That's **3 separate matmuls**. Each one re-reads x from VRAM. Unsloth does it in a single fused kernel.

**`cross_entropy_loss.py` (463 lines) — Fused CE**

Standard cross-entropy:
```
logits = model(x)            # [batch, seq, vocab] — held in VRAM
log_probs = log_softmax(logits)
loss = -log_probs[targets]
```
With a vocab of ~150K, the `logits` tensor is huge (`batch * seq * 150K * 2 bytes` in bf16). At 2K context that's 600 MB!

Unsloth's fused CE: `logits` is never materialized — `log_softmax + nll` is done in one kernel. **VRAM saving: ~50%**.

**`rms_layernorm.py` (339 lines) — Fused RMSNorm**

Llama, Qwen, and Mistral all use RMSNorm. PyTorch's standard implementation is 4-5 separate kernels:
```
var = x.pow(2).mean(-1)      # kernel 1: square + reduce
rstd = 1 / sqrt(var + eps)    # kernel 2: rsqrt
y = x * rstd                  # kernel 3: multiply
y = y * weight                # kernel 4: scale
```
Unsloth uses one kernel — computes everything in registers, never writes intermediates.

**`rope_embedding.py` (465 lines) — Fused RoPE**

Rotary Position Embedding (RoPE) recomputes trig functions every layer. Unsloth precomputes and applies them in a fused kernel, dropping that cost.

**`swiglu.py` (143 lines) — Fused SwiGLU**

The modern FFN: `silu(gate(x)) * up(x) → down(...)`. In PyTorch: 3 matmuls + activation + multiply. Unsloth fuses it.

### MoE Kernels (new in 2026)

A separate module under `kernels/moe/`. A grouped GEMM that handles sparse routing at the kernel level. Covered in detail in Notebook 05 of this guide.

### Fused Losses (`unsloth_zoo/fused_losses/`)

Separate fused loss kernels for DPO, ORPO, KTO, GRPO. Covered in Notebooks 06-07.

In [ ]:
# Look inside one kernel
import os

kernel_path = os.path.join(
    os.path.dirname(__import__('unsloth').__file__),
    'kernels', 'rms_layernorm.py'
)

with open(kernel_path) as f:
    content = f.read()

# Find @triton.jit decorators
import re
kernels = re.findall(r'@triton\.(jit|autotune)[\s\S]*?def (\w+)', content)
print(f'Triton kernels in rms_layernorm.py:')
for kind, name in kernels:
    print(f'  @{kind}: {name}')

# Show the signature of the first kernel
if kernels:
    first_name = kernels[0][1]
    pattern = re.compile(rf'def {first_name}\(([\s\S]*?)\):')
    m = pattern.search(content)
    if m:
        print(f'\n{first_name} signature:\n{m.group(0)}')

## 4. Manual Backprop — Bypassing `autograd`

PyTorch's autograd records the forward graph and walks back through the computational graph during backward. **Convenient, but slow**:
1. Every forward op creates a graph node
2. Intermediate tensors are kept in VRAM (needed for backward)
3. Backward is generic — no operation-specific optimization possible

Unsloth writes manual backprop using `torch.autograd.Function`:

```python
class FastRMSNorm(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, weight, eps):
        # 1. Compute RMSNorm via the Triton kernel
        y = _rms_layernorm_forward_kernel(x, weight, eps)
        # 2. Save only the tensors needed for backward
        ctx.save_for_backward(x, weight)
        ctx.eps = eps
        return y

    @staticmethod
    def backward(ctx, grad_output):
        # 3. Manual backward — fused Triton kernel
        x, weight = ctx.saved_tensors
        grad_x, grad_w = _rms_layernorm_backward_kernel(
            grad_output, x, weight, ctx.eps
        )
        return grad_x, grad_w, None
```

**Advantages:**
1. Backward lives in its own kernel — fused and optimized
2. Only the tensors that are truly needed get saved (autograd saves everything)
3. You can use operation-specific math (e.g. skip a transpose if `x` is symmetric)

### Numerical Accuracy

Unsloth's claim: **0% accuracy loss**. Because:
- The op is mathematically identical
- bf16 ↔ fp32 conversions stay within standard bounds
- The manual backward derivatives are mathematically verified

The only difference is the **execution path**. The output tensors are byte-for-byte identical.

In [ ]:
# Look at Unsloth's autograd.Function classes
import os, re

kernels_dir = os.path.join(os.path.dirname(__import__('unsloth').__file__), 'kernels')

for f in os.listdir(kernels_dir):
    if not f.endswith('.py') or f.startswith('_'):
        continue
    path = os.path.join(kernels_dir, f)
    with open(path) as fh:
        content = fh.read()
    # Find autograd.Function classes
    classes = re.findall(r'class (\w+)\(.*?(?:autograd\.Function|Function).*?\):', content)
    if classes:
        print(f'{f}:')
        for c in classes:
            print(f'  {c}')

## 5. Model Registry — Detection Mechanism

When Unsloth loads a model it **automatically detects what kind it is** and applies the appropriate patches. The mechanism:

### Registries

**`unsloth/models/mapper.py`** — maps a model name to its canonical model:
```python
INT_TO_FLOAT_MAPPER = {
    'unsloth/llama-3-8b-bnb-4bit': 'meta-llama/Meta-Llama-3-8B',
    'unsloth/qwen3-1.7b-bnb-4bit': 'Qwen/Qwen3-1.7B',
    # ...
}
```
Thanks to this mapping, when you load a 4-bit Unsloth variant it knows the base model name and applies the right patches.

**`FORCE_FLOAT32`** — for some model types bf16 causes numerical issues; this list forces them to FP32.

**`DISABLE_SDPA_MODEL_NAMES`** — models where Scaled Dot-Product Attention is buggy; falls back to flash attention or a custom attention implementation.

### Detection Flow

```
FastModel.from_pretrained('Qwen/Qwen3-1.7B-Base')
  ↓
1. Read model_type via AutoConfig → 'qwen3'
  ↓
2. Look at the architecture → 'Qwen3ForCausalLM'
  ↓
3. Decision tree:
     - vision_config present? → text-only
     - 'MoE' suffix? → standard text
     - Present in INT_TO_FLOAT_MAPPER? → apply mapping
  ↓
4. Load unsloth/models/qwen3.py
  ↓
5. Activate Qwen3-specific patches:
     - Qwen3DecoderLayer.forward → fused
     - Qwen3RMSNorm.forward → Triton
     - etc.
  ↓
6. Load the model (HF normal flow)
  ↓
7. Apply post-load patches (for peft layers)
```

### Why `FastModel` Is An Improvement

Old `FastLanguageModel` assumed Llama-style models. `FastModel` routes by `model_type`:
- `qwen3` → `qwen3.py`
- `llama` → `llama.py`
- `mistral` → `mistral.py`
- `qwen3_moe` → `qwen3_moe.py` (MoE-specific)

In [ ]:
# Which models have specific patches?
import os

models_dir = os.path.join(os.path.dirname(__import__('unsloth').__file__), 'models')
model_files = sorted([f for f in os.listdir(models_dir) if f.endswith('.py') and not f.startswith('_')])
print('Model-specific patches:')
for f in model_files:
    path = os.path.join(models_dir, f)
    with open(path) as fh:
        lines = sum(1 for _ in fh)
    print(f'  {f:30s} {lines:5d} lines')

## 6. Memory Savings — The Math of Fused Operations

**Where does Unsloth's 70% less VRAM come from?** Three main sources:

### A. Activation Memory (the biggest contributor)

During forward, every layer keeps its activations in VRAM (for backward). Standard PyTorch:

```
Layer 1 input: x₁  ← saved
Layer 1 output: y₁ = f(x₁)  ← saved (for backward)
Layer 2 input: x₂ = y₁  ← saved
Layer 2 output: y₂ = g(x₂)  ← saved
...
```

Two activations per layer. In a 32-layer model that's **64 activation tensors**.

**Unsloth gradient checkpointing (`use_gradient_checkpointing='unsloth'`)** keeps only activations at checkpoint positions and **recomputes the rest during backward**.

Standard PyTorch gradient checkpointing does the same thing, but Unsloth's version:
- Can also offload to CPU (async)
- Recomputes activations using a Triton kernel (faster recompute)

### B. Logits/Loss Memory (thanks to the CE kernel)

Standard cross-entropy:
```
logits: [batch, seq, vocab] = [4, 2048, 150000]  → 2.4 GB (bf16)
log_softmax(logits): same shape → another 2.4 GB
Total: ~5 GB just for the loss
```

Fused CE kernel:
- The `logits` tensor is **never materialized**
- log_softmax is computed tile-by-tile, only the value at the target index is taken
- VRAM: ~150 MB (orders of magnitude less)

### C. LoRA Adapter Memory

Standard PEFT LoRA:
```
x → x_lora_in → A @ x_lora_in → B @ ... → x_lora_out
      (saved)         (saved)            (saved)
```

Unsloth fused LoRA: computes it in a single pass, no intermediates stored.

### Numerical Comparison

Llama-3-8B QLoRA, batch=2, seq=2048:

| Component | PyTorch + PEFT | Unsloth | Savings |
|-----------|----------------|---------|---------|
| Model weights (4-bit) | 4.5 GB | 4.5 GB | 0% |
| LoRA adapters | 0.05 GB | 0.05 GB | 0% |
| Activations | 8.0 GB | 2.5 GB | -69% |
| Logits (CE) | 5.0 GB | 0.15 GB | -97% |
| Optimizer state (8-bit AdamW) | 0.4 GB | 0.4 GB | 0% |
| Gradients | 0.5 GB | 0.5 GB | 0% |
| **Total (peak)** | **~18 GB** | **~8 GB** | **-56%** |

These are approximate — actual numbers vary with model/seq/batch.

## 7. `use_gradient_checkpointing='unsloth'` In Detail

Standard options:
- `False` — checkpointing off, fastest but VRAM-hungry
- `True` — PyTorch's standard checkpointing
- `'unsloth'` — Unsloth's variant, more aggressive

### Standard PyTorch Checkpointing

During the forward pass it **drops** activations at certain layers. During backward, when it reaches that layer, it **re-runs the forward**.

```
Forward:  layer 1 → save → layer 2 → drop → layer 3 → save → ...
Backward: ... layer 3 done → layer 2 grad needed → re-run forward
```

Trade-off: 30-50% VRAM savings, 20-30% speed loss.

### Unsloth Checkpointing

Three additional optimizations:

**1. CPU Offload (async)**
Instead of dropping activations, send them to CPU RAM. Pull them back during backward. Standard overlap pattern:
```
Forward: layer 2 done
  └─→ async copy activation to CPU (overlap with layer 3 forward)
Backward: layer 2 grad needed
  └─→ async copy from CPU (overlap with layer 3 backward)
```

If PCIe bandwidth is sufficient, the speed loss is minimal.

**2. Triton Recompute Kernels**
Use Triton kernels when recomputing dropped layers. ~30% faster recompute than standard PyTorch.

**3. Smart Drop Selection**
Cost-aware choice of what to drop:
- Expensive layers (dense FFN) → kept
- Cheap layers (norm, residual) → dropped

### In Practice

How does this play out per model?

| Model | Checkpoint = False | True | 'unsloth' |
|-------|--------------------|------|-----------|
| Llama-3-8B (LoRA) | 28 GB | 22 GB | **18 GB** |
| Qwen3-32B (QLoRA) | 60 GB | 38 GB | **24 GB** |
| Speed (relative) | 1.0× | 0.75× | **0.85×** |

So `'unsloth'` uses less VRAM than standard True **and** is faster.

## 8. Debug — Knowing the Internals Makes Errors Easier

### Error 1: `ImportError: please install transformers via pip install`
Cause: Unsloth is trying to patch `transformers` but can't find it.

### Error 2: `RuntimeError: PASSING ZERO LENGTH TENSOR ...`
Cause: A Triton kernel was called with an empty tensor. Usually a batch=0 edge case.

### Error 3: `KeyError: 'qwen3_5'`
Cause: Unknown model_type in the registry. Your transformers is too old.

### Error 4: `Unsloth: Llama doesn't have rotary embedding cached!`
Cause: The RoPE cache wasn't patched. `import unsloth` is probably not at the top.

### Error 5: `ZeroDivisionError: All labels in your dataset are -100`
Cause: The fused CE kernel can't compute loss over 0 tokens (division). The mask covered the entire sequence.

### Pattern
Check the imports — wrong order?
Is the model recognized in the registry — `print(model.config.model_type)`
Are the patches active — are `unsloth.__version__` and `transformers.__version__` compatible?
Is it a Triton compile error — re-run with `TRITON_INTERPRET=1`

## Summary

After this notebook:

✅ You know Unsloth's two packages (`unsloth` + `unsloth_zoo`) and the layers inside them
✅ You know why `import unsloth` goes first — the patching mechanism is clear
✅ You know which Triton kernels exist and what they optimize
✅ You understand why manual backprop is needed and the trade-offs of autograd
✅ You understand the model registry — `model_type` → specific patch path
✅ You know the math behind the memory savings — activations, CE, LoRA
✅ You know what `use_gradient_checkpointing='unsloth'` actually does

**Next notebook:** `02_quantization.ipynb` — `bnb-4bit` vs `unsloth-bnb-4bit` (Dynamic 2.0) in depth, and when to use which quant.